In [2]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

In [1]:
import base64
import os
from io import BytesIO
from PIL import Image

from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

In [3]:
# --- 1. Gemini 비전 모델을 활용한 LangChain 도구 정의 ---
# @tool 데코레이터를 사용하여 간단하게 도구를 생성합니다.
@tool
def image_analyzer(image_path: str, query: str) -> str:
    """
    지정된 경로의 이미지를 분석하고 사용자의 질문에 답변합니다.
    이미지 경로(image_path)와 질문(query)을 인자로 받습니다.
    """
    # 로컬 이미지 파일을 읽어 Base64로 인코딩합니다.
    try:
        with open(image_path, "rb") as image_file:
            encoded_image = base64.b64encode(image_file.read()).decode("utf-8")
        image_data = f"data:image/jpeg;base64,{encoded_image}"
    except FileNotFoundError:
        return f"오류: {image_path} 에서 파일을 찾을 수 없습니다."

    # Gemini 비전 모델을 초기화합니다.
    # 사용자가 'gemini-2.5-flash image'를 요청했지만,
    # LangChain에서는 'gemini-pro-vision'이 비전 작업에 사용되는 공식 모델명입니다.
    llm = ChatGoogleGenerativeAI(model="gemini-pro-vision")

    # 모델에 전달할 메시지를 구성합니다. (텍스트 + 이미지)
    message = HumanMessage(
        content=[
            {"type": "text", "text": query},
            {"type": "image_url", "image_url": image_data},
        ]
    )

    # 모델을 호출하고 결과를 반환합니다.
    response = llm.invoke([message])
    return response.content

In [ ]:
# # --- 2. 이미지 생성 및 준비 (예시) ---
# # 실제 사용 시에는 이 부분을 분석하려는 이미지 경로로 대체하면 됩니다.
# try:
#     img = Image.new('RGB', (200, 100), color = 'red')
#     img.save('example-image.jpg')
#     print("'example-image.jpg' 파일을 생성했습니다.")
# except Exception as e:
#     print(f"예시 이미지 생성 중 오류 발생: {e}")

In [4]:
# --- 3. 에이전트 설정 및 실행 ---
# 사용할 도구들을 리스트로 묶습니다.
tools = [image_analyzer]

# ReAct 프롬프트 템플릿을 가져옵니다.
# ReAct는 모델이 생각(Thought)과 행동(Action)을 통해 추론하는 과정을 돕는 프롬프트 엔지니어링 기법입니다.
prompt = hub.pull("hwchase17/react")

# 사용할 언어 모델을 다시 초기화합니다. (이번에는 텍스트 전용)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 에이전트를 생성합니다.
agent = create_react_agent(llm, tools, prompt)

# 에이전트 실행기를 만듭니다.
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


# --- 4. 에이전트에게 작업 지시 ---
# 에이전트에게 이미지 분석을 요청합니다.
# image_analyzer 도구가 호출되는 과정을 확인할 수 있습니다.
result = agent_executor.invoke({
    "input": "'bg.jpg' 파일은 어떤 색상의 이미지인가요?"
})

# 최종 결과 출력
print("\n[최종 답변]:")
print(result["output"])



> Entering new AgentExecutor chain...


ChatGoogleGenerativeAIError: Invalid argument provided to Gemini: 400 API key expired. Please renew the API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key expired. Please renew the API key."
]